# The Interpretability Agenda

## What This Is
Mechanistic interpretability (mech interp) aims to reverse-engineer neural networks: understand not just *what* they compute but *how* they implement computations in their weights and activations.

## Feature Visualisation History (Narrative)
Early interpretability used feature visualisation: synthesize an input that maximally activates a neuron. Zeiler & Fergus (2014) used deconvnets; Simonyan et al. used gradient ascent on the input. These visualisations showed that early CNN layers learn edges and textures while later layers learn object parts.

However, feature visualisation has limitations:
- Many neurons are *polysemantic* — they activate for multiple unrelated concepts
- Visualisations can be deceiving — adversarially crafted images can look like one thing but activate like another
- Doesn't reveal circuit-level computations

## Mechanistic Interpretability Goals
- **Features**: What concepts does the model represent internally?
- **Circuits**: How does information flow between components to implement an algorithm?
- **Universality**: Do the same circuits appear across different models?

Anthropologic research (Anthropic, Olah et al., 2020-present) has found that many of the same feature types (curve detectors, multimodal neurons) appear in both biological and artificial neural networks.

In [ ]:
import numpy as np
from typing import Dict, List, Tuple, Optional

np.random.seed(42)

# Build a small transformer for mech interp demos
# Architecture: 2-layer, 2-head attention, d_model=16

D_MODEL = 16
N_HEADS = 2
D_HEAD = D_MODEL // N_HEADS
VOCAB_SIZE = 50  # tiny vocabulary
SEQ_LEN = 8

class TinyTransformer:
    def __init__(self, seed=42):
        np.random.seed(seed)
        # Embeddings
        self.W_embed = np.random.randn(VOCAB_SIZE, D_MODEL) * 0.1
        self.W_pos = np.random.randn(SEQ_LEN, D_MODEL) * 0.1
        
        # Layer 0
        self.W_Q0 = [np.random.randn(D_MODEL, D_HEAD) * 0.1 for _ in range(N_HEADS)]
        self.W_K0 = [np.random.randn(D_MODEL, D_HEAD) * 0.1 for _ in range(N_HEADS)]
        self.W_V0 = [np.random.randn(D_MODEL, D_HEAD) * 0.1 for _ in range(N_HEADS)]
        self.W_O0 = np.random.randn(N_HEADS * D_HEAD, D_MODEL) * 0.1
        self.W_ff0 = np.random.randn(D_MODEL, 4 * D_MODEL) * 0.1
        self.W_ff0_out = np.random.randn(4 * D_MODEL, D_MODEL) * 0.1
        
        # Layer 1
        self.W_Q1 = [np.random.randn(D_MODEL, D_HEAD) * 0.1 for _ in range(N_HEADS)]
        self.W_K1 = [np.random.randn(D_MODEL, D_HEAD) * 0.1 for _ in range(N_HEADS)]
        self.W_V1 = [np.random.randn(D_MODEL, D_HEAD) * 0.1 for _ in range(N_HEADS)]
        self.W_O1 = np.random.randn(N_HEADS * D_HEAD, D_MODEL) * 0.1
        self.W_ff1 = np.random.randn(D_MODEL, 4 * D_MODEL) * 0.1
        self.W_ff1_out = np.random.randn(4 * D_MODEL, D_MODEL) * 0.1
        
        # Unembedding
        self.W_unembed = np.random.randn(D_MODEL, VOCAB_SIZE) * 0.1
        
        self.activations = {}  # cache for interpretability
    
    def softmax(self, x, axis=-1):
        ex = np.exp(x - x.max(axis=axis, keepdims=True))
        return ex / ex.sum(axis=axis, keepdims=True)
    
    def attention(self, x, WQ, WK, WV):
        T, d = x.shape
        Q = x @ WQ; K = x @ WK; V = x @ WV
        scores = Q @ K.T / np.sqrt(D_HEAD)
        mask = np.triu(np.ones((T, T)) * -1e9, k=1)
        attn = self.softmax(scores + mask)
        return attn @ V, attn
    
    def forward(self, token_ids: List[int]):
        T = len(token_ids)
        x = self.W_embed[token_ids] + self.W_pos[:T]
        self.activations['embed'] = x.copy()
        
        # Layer 0
        heads = []
        attns = []
        for h in range(N_HEADS):
            hv, attn = self.attention(x, self.W_Q0[h], self.W_K0[h], self.W_V0[h])
            heads.append(hv); attns.append(attn)
        attn_out = np.concatenate(heads, axis=-1) @ self.W_O0
        x = x + attn_out
        x = x + np.maximum(0, x @ self.W_ff0) @ self.W_ff0_out
        self.activations['layer0'] = x.copy()
        self.activations['attn0'] = attns
        
        # Layer 1
        heads = []
        attns1 = []
        for h in range(N_HEADS):
            hv, attn = self.attention(x, self.W_Q1[h], self.W_K1[h], self.W_V1[h])
            heads.append(hv); attns1.append(attn)
        attn_out = np.concatenate(heads, axis=-1) @ self.W_O1
        x = x + attn_out
        x = x + np.maximum(0, x @ self.W_ff1) @ self.W_ff1_out
        self.activations['layer1'] = x.copy()
        self.activations['attn1'] = attns1
        
        logits = x @ self.W_unembed
        self.activations['logits'] = logits.copy()
        return logits

model = TinyTransformer()
tokens = [1, 5, 12, 3, 8, 20, 7, 4]
logits = model.forward(tokens)
print(f'Model output shape: {logits.shape} (seq_len={len(tokens)}, vocab={VOCAB_SIZE})')
print(f'Activation keys: {list(model.activations.keys())}')
print(f'Layer 0 residual shape: {model.activations["layer0"].shape}')
print(f'Attention pattern (head 0, layer 0):')
print(model.activations['attn0'][0].round(3))
